# 41 — Design · PM · Sales · CS · Marketing

The non-engineering side of the specialist roster. Each persona is prompted for domain-correct outputs — a design reviewer leads with user tasks, a marketing writer cuts adjectives, a CS manager opens with the customer's stated success metric.

Runs on **Bedrock Llama** — no extra credentials required.

In [ ]:
from pathlib import Path
import sys

# Resolve the repo root whether this cell is running from ./notebooks
# or from the repo root — mirrors the existing 01–36 notebook series.
ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.run_multi_tool_agent import build_llm_from_env

# Bedrock Llama 4 Scout is the default model this library ships with
# (see `shipit_agent/config.py`). No model name required — the helper
# reads `AWS_REGION` / credentials from env and returns an adapter that
# works with every Autopilot / Agent class.
llm = build_llm_from_env("bedrock")
print("Bedrock LLM ready:", type(llm).__name__)


In [ ]:
from shipit_agent import Agent
from shipit_agent.agents import AgentRegistry
from shipit_agent.builtins import get_builtin_tools
registry = AgentRegistry()


## 1. Design reviewer — user-task first, accessibility non-negotiable

In [ ]:
design_def = registry.get("design-reviewer")
designer = Agent(
    llm=llm, prompt=design_def.prompt,
    tools=get_builtin_tools(project_root="."),
    max_iterations=design_def.maxIterations or 20,
    name=design_def.name,
)
result = designer.run("Review the onboarding flow described in this repo's README and list P0/P1/P2 findings with the principle each violates.")
print(result.output[:900])


## 2. Product manager — scope minimally, specify non-goals

The PM specialist is good at turning a vague ask into a shippable MVP + explicit non-goals. Feed it a one-line feature request and you get a structured spec back.

In [ ]:
pm_def = registry.get("product-manager")
pm = Agent(
    llm=llm, prompt=pm_def.prompt, tools=[], name=pm_def.name,
    max_iterations=pm_def.maxIterations or 20,
)
result = pm.run("Spec out a lightweight 'share run' feature for Autopilot that lets engineers email a read-only link to a completed run.")
print(result.output[:900])


## 3. Sales outreach — research first, then a 120-word email

In [ ]:
sales_def = registry.get("sales-outreach")
sales = Agent(
    llm=llm, prompt=sales_def.prompt, tools=[], name=sales_def.name,
    max_iterations=sales_def.maxIterations or 15,
)
result = sales.run("Prep an outreach for the VP of Engineering at a Series B fintech that just announced a hiring freeze. Our pitch is unattended overnight refactoring runs that cut eng costs.")
print(result.output[:900])


### 3a. Wire in HubSpot for real CRM actions

If you have a HubSpot private-app token in `HUBSPOT_TOKEN`, the `hubspot_ops` tool gives the sales agent search / create / add-note superpowers against your real CRM.

In [ ]:
import os

if os.environ.get("HUBSPOT_TOKEN"):
    from shipit_agent.tools.hubspot import HubspotTool
    sales_with_crm = Agent(
        llm=llm, prompt=sales_def.prompt,
        tools=[HubspotTool()],
        max_iterations=12,
        name=sales_def.name,
    )
    print("Sales agent ready with HubSpot tool — replace the .run() call with your actual prospect context.")
else:
    print("Set HUBSPOT_TOKEN in your env to enable live CRM lookups — this cell is a no-op without it.")


## 4. Customer success — onboarding, health, renewal

CS agent's super-power is reading before writing: sales handoff notes + usage signals + ticket history, then a structured 30-day plan.

In [ ]:
cs_def = registry.get("customer-success")
cs = Agent(
    llm=llm, prompt=cs_def.prompt, tools=[], name=cs_def.name,
    max_iterations=cs_def.maxIterations or 15,
)
result = cs.run("A new $150k/yr customer just signed. Their stated goal was 'cut incident response time by 40%'. Draft the first 30-day success plan and kickoff agenda.")
print(result.output[:900])


## 5. Marketing writer — concrete, not clever

In [ ]:
mkt_def = registry.get("marketing-writer")
mkt = Agent(
    llm=llm, prompt=mkt_def.prompt, tools=[], name=mkt_def.name,
    max_iterations=mkt_def.maxIterations or 15,
)
result = mkt.run("Write the launch post for shipit_agent 1.1 — the Autopilot release. Readers are senior engineers. Keep it ≤250 words, concrete, no 'blazing fast'.")
print(result.output[:900])


## Summary

Five specialists you can drop into a Slack bot, a Notion task, or wrap in Autopilot for long-form artifacts. Every prompt was written as a production spec — not a sales demo — so outputs tend to be usable as-is.

Next: **42 — computer_use and the other power tools**.